# Module 09 - 실습 3 솔루션: Health Check 에이전트

In [ ]:
import sys
sys.path.insert(0, '../..')

import asyncio
import time
from enum import Enum
from dataclasses import dataclass

print("Health Check 솔루션 준비 완료")

## 실습 1 솔루션: HealthStatus Enum

In [ ]:
class HealthStatus(Enum):
    HEALTHY = "healthy"
    DEGRADED = "degraded"    # 일부 시스템만 장애
    UNHEALTHY = "unhealthy"

In [ ]:
# 검증
assert HealthStatus.HEALTHY.value == "healthy"
assert HealthStatus.DEGRADED.value == "degraded"
assert HealthStatus.UNHEALTHY.value == "unhealthy"
print("✅ 실습 1 완료! HealthStatus Enum이 올바르게 정의되었습니다.")

## 실습 2 솔루션: SystemHealth dataclass

In [ ]:
@dataclass
class SystemHealth:
    """개별 시스템의 상태."""
    name: str
    status: HealthStatus
    latency_ms: float | None = None
    error: str | None = None

In [ ]:
# 검증
health = SystemHealth(name="redis", status=HealthStatus.HEALTHY, latency_ms=12.5)
assert health.name == "redis"
assert health.status == HealthStatus.HEALTHY
assert health.latency_ms == 12.5
assert health.error is None
print("✅ 실습 2 완료! SystemHealth dataclass가 올바르게 정의되었습니다.")

## 실습 3 솔루션: check_service 함수

In [ ]:
SERVICE_CONFIG = {
    "redis": {"delay": 0.05, "fail": False},
    "jira": {"delay": 0.1, "fail": False},
    "gitlab": {"delay": 0.08, "fail": True},
}

async def check_service(name: str) -> SystemHealth:
    """서비스 헬스체크를 수행합니다 (시뮬레이션)."""
    config = SERVICE_CONFIG.get(name, {"delay": 0.1, "fail": False})
    
    start = asyncio.get_event_loop().time()
    try:
        await asyncio.sleep(config["delay"])  # 네트워크 지연 시뮬레이션
        latency = (asyncio.get_event_loop().time() - start) * 1000
        
        if config["fail"]:
            return SystemHealth(
                name=name,
                status=HealthStatus.UNHEALTHY,
                latency_ms=latency,
                error=f"Connection refused to {name}",
            )
        
        return SystemHealth(
            name=name,
            status=HealthStatus.HEALTHY,
            latency_ms=latency,
        )
    except Exception as exc:
        return SystemHealth(
            name=name,
            status=HealthStatus.UNHEALTHY,
            error=str(exc),
        )

In [ ]:
redis_health = await check_service("redis")
print(f"Redis: {redis_health}")

gitlab_health = await check_service("gitlab")
print(f"GitLab: {gitlab_health}")

In [ ]:
# 검증
assert isinstance(redis_health, SystemHealth)
assert redis_health.status == HealthStatus.HEALTHY
assert redis_health.latency_ms is not None
assert gitlab_health.status == HealthStatus.UNHEALTHY
print("✅ 실습 3 완료! check_service가 올바르게 작동합니다.")

## 실습 4 솔루션: run_health_check 함수

In [ ]:
async def run_health_check() -> dict:
    """모든 시스템의 상태를 확인합니다."""
    # 모든 서비스를 동시에 체크
    checks = await asyncio.gather(
        check_service("redis"),
        check_service("jira"),
        check_service("gitlab"),
        return_exceptions=True,
    )
    
    results = []
    for check in checks:
        if isinstance(check, Exception):
            results.append(SystemHealth("unknown", HealthStatus.UNHEALTHY, error=str(check)))
        else:
            results.append(check)
    
    # 전체 상태 판단
    unhealthy = [r for r in results if r.status == HealthStatus.UNHEALTHY]
    
    if len(unhealthy) == 0:
        overall = HealthStatus.HEALTHY
    elif len(unhealthy) < len(results):
        overall = HealthStatus.DEGRADED
    else:
        overall = HealthStatus.UNHEALTHY
    
    return {
        "status": overall.value,
        "systems": [
            {
                "name": r.name,
                "status": r.status.value,
                "latency_ms": r.latency_ms,
                "error": r.error,
            }
            for r in results
        ],
    }

In [ ]:
import json
start = time.time()
health_report = await run_health_check()
elapsed = time.time() - start

print(json.dumps(health_report, indent=2, ensure_ascii=False))
print(f"\n헬스체크 완료 시간: {elapsed:.2f}초")

In [ ]:
# 검증
assert "status" in health_report
assert "systems" in health_report
assert len(health_report["systems"]) == 3
assert health_report["status"] == HealthStatus.DEGRADED.value
assert elapsed < 0.5
print(f"✅ 실습 4 완료! run_health_check가 올바르게 동작합니다. (전체 상태: {health_report['status']})")

## 정리

이번 실습에서 배운 내용:
1. `Enum`으로 상태를 명확하게 표현할 수 있다
2. `dataclass`로 구조화된 데이터를 정의할 수 있다
3. `asyncio.gather`로 여러 시스템을 동시에 체크하여 빠르게 헬스체크할 수 있다